In [ ]:
# RESULTS = "KEEP/toy_FULL_1407/all_results.csv"
RESULTS_MAP = "results_map"
EXP_FOLDER = "KEEP/toy_FULL_1407/"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from src.config import Config
from notebooks.analysis_utils import ModelLabels, CustPalette, heatmap_df

# Data aggregation

In [ ]:
util_cost_col_name = "Util Cost"
ppv_uplift_col_name = "PPV Uplift"
recall_uplift_col_name = "Recall Uplift"
f1_uplift_col_name = "F1 Uplift"
eor_delta_col_name = "EOR Delta"

In [ ]:
with open(f"{PROJECT_ROOT}/configs/{RESULTS_MAP}.json", 'r') as f:
  results_map = json.load(f)

def calc_avg_perc(metric):
  return round(metric.mean()*100, 2)

all_results = []

for r in results_map:
  dataset_str = f"prev{r['prev']}_dim{r['dim']}_{r['type']}"
  scm_map_id = f"toy_{r['dim']}_{r['scm']}"
  try:
    results = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_FOLDER}/{dataset_str}_FULL_{r['scm']}/bootstrapped_evaluation_results.csv')
  except:
    continue

  results['model'] = results['model'].replace({
    'baseline': ModelLabels.baseline,
    'baseline_unaware': ModelLabels.baseline_unaware,
    'ablation': ModelLabels.ablation,
    'fair': ModelLabels.fair,
    'fair_unaware': ModelLabels.fair_unaware
  })
  results.sort_values(by='model', axis=0, inplace=True)

  results['f1'] = 2*results['recall']*results['ppv'] / (results['recall'] + results['ppv'])


  # GLOBAL AUPRC
  global_results = results[results['subgroup'] == "Global"]

  agg_results = global_results.groupby('model')['auprc'].agg(calc_avg_perc).to_frame(name="auprc")

  # IDENTIFIERS
  agg_results = agg_results.assign(**r)
  agg_results['dataset'] = dataset_str

  # ===============
  # GROUP RESULTS
  # ===============
  group_results = results[results['subgroup'] != "Global"].pivot_table(index=['model', 'bootstrap_idx'], columns='subgroup', values=['auprc', 'recall', 'ppv', 'f1', 'fpr'], aggfunc="mean")
  
  # Add equalised odds to the results
  lowest_tpr = group_results[[('recall','Group 0'), ('recall','Group 1')]].min(axis=1)
  highest_tpr = group_results[[('recall','Group 0'), ('recall','Group 1')]].max(axis=1)
  tpr_ratio = lowest_tpr / highest_tpr 
  lowest_fpr = group_results[[('fpr','Group 0'), ('fpr','Group 1')]].min(axis=1)
  highest_fpr = group_results[[('fpr','Group 0'), ('fpr','Group 1')]].max(axis=1)
  fpr_ratio = lowest_fpr / highest_fpr 
  group_results['equalised odds ratio'] = np.minimum(tpr_ratio, fpr_ratio)

  # Add minimum group metrics
  group_results['min recall'] = lowest_tpr
  group_results['min ppv'] = group_results[[('ppv','Group 0'), ('ppv','Group 1')]].min(axis=1)
  group_results['min f1'] = group_results[[('f1','Group 0'), ('f1','Group 1')]].min(axis=1)

  group_results = group_results.groupby('model').agg(calc_avg_perc)

  group_results.columns = [
    f"{metric} - {subgroup}" if pd.notna(subgroup) and subgroup != '' else f"{metric}"
    for metric, subgroup in group_results.columns
  ]

  agg_results = agg_results.merge(group_results, right_index=True, left_index=True)


  # ======
  # Relative metrics to baseline
  # ======
  baseline_row = agg_results.loc[ModelLabels.baseline, :]
  def metric_diff(row, metric, relative=True, gain=True):
    diff = (row[metric] - baseline_row[metric]) if gain else (baseline_row[metric] - row[metric])
    if relative:
      return diff / baseline_row[metric] * 100
    else: 
      return diff
  
  agg_results[util_cost_col_name] = agg_results.apply(
    lambda x: metric_diff(x, "auprc", relative=True, gain=False), axis=1)
  
  agg_results[ppv_uplift_col_name] = agg_results.apply(
    lambda x: metric_diff(x, "min ppv", relative=True, gain=False), axis=1)
  
  agg_results[recall_uplift_col_name] = agg_results.apply(
    lambda x: metric_diff(x, "min recall", relative=True, gain=False), axis=1)
  
  agg_results[f1_uplift_col_name] = agg_results.apply(
    lambda x: metric_diff(x, "min f1", relative=True, gain=False), axis=1)
  
  agg_results[eor_delta_col_name] = agg_results.apply(
    lambda x: metric_diff(x, "equalised odds ratio", relative=True, gain=False), axis=1)

  agg_results['run'] = f"{dataset_str}_{r["scm"]}"
  agg_results.reset_index(inplace=True, names="Model")
  all_results.append(agg_results)

all_results_df = pd.concat(all_results, ignore_index=True)


all_results_df['Pathway Config'] = all_results_df['config'].replace({
  1: "Ground Truth",
  2: "Min F1 Uplift (FEU)",
  3: "Min F1 Uplift (Dist)",
  4: "S Info Reduc (FEU)",
  5: "S Info Reduc (Dist)",
  6: "CF Harm Reduc (FEU)",
  7: "CF Harm Reduc (Dist)"
})

all_results_df = all_results_df[all_results_df['Model'] != ModelLabels.baseline]

# print(all_results_df)

In [ ]:
all_results_df[['Model','Pathway Config']].value_counts()

# Gain/Cost Comparison Across Runs

## Plotting functions

In [ ]:
import textwrap

def parallel_coord_plot(metric, title, gain=True, col="Pathway Config", hue="dim"):
    y_data_min = all_results_df[metric].min()
    y_data_max = all_results_df[metric].max()
    y_range = y_data_max - y_data_min if y_data_max != y_data_min else 1.0
    margin = 0.10 * y_range
    global_ymin = y_data_min - margin
    global_ymax = y_data_max + margin

    g = sns.relplot(
        data=all_results_df,
        x="Model",
        y=metric,
        col=col,
        hue=hue,
        units="run",  
        estimator=None,
        kind="line",
        col_wrap=2,
        linewidth=1.5,
        legend=True,
    )

    idx_min = all_results_df.groupby(["run", "Pathway Config"])[metric].idxmin()
    df_min = all_results_df.loc[idx_min]
    # print(df_min)
    idx_max = all_results_df.groupby(["run", "Pathway Config"])[metric].idxmax()
    df_max = all_results_df.loc[idx_max]

    for col_value, ax in g.axes_dict.items():
        ax.set_ylim(global_ymin, global_ymax)
        if gain:
            ax.axhspan(0, max(global_ymax, 0.01), color="green", alpha=0.15, zorder=0)
        else:
            ax.axhspan(min(global_ymin, -0.01), 0, color="green", alpha=0.15, zorder=0)

        config_min = df_min[df_min[col] == col_value]
        # print(config_min)
        config_max = df_max[df_max[col] == col_value]
        sns.scatterplot(
            data=config_min,
            x="Model",
            y=metric,
            color="red" if gain else "green",
            marker="v",
            s=100,
            ax=ax,
            zorder=5,
            legend=False
        )
        sns.scatterplot(
            data=config_max,
            x="Model",
            y=metric,
            color="green" if gain else "red",
            marker="^",
            s=100,
            ax=ax,
            zorder=5,
            legend=False
        )
        ax.tick_params(rotation=30)

    # X LABELS
    labels = [item.get_text() for item in ax.get_xticklabels()]
    wrapped_labels = [textwrap.fill(label, width=15) for label in labels]

    ax.set_xticks(ax.get_xticks())
    ax.set_xticklabels(wrapped_labels, rotation=0, rotation_mode="anchor")
    
    # LEGEND
    handles = g._legend.legend_handles
    labels = [t.get_text() for t in g._legend.get_texts()]

    # 3. Create two invisible plots on the current axis just to generate legend entries
    max_handle = plt.Line2D(
        [],
        [],
        marker="^",
        color="green" if gain else "red",
        ls="",
        markersize=8,
    )
    min_handle = plt.Line2D(
        [],
        [],
        marker="v",
        color="red" if gain else "green",
        ls="",
        markersize=8,
    )

    # 4. Combine them together
    handles.extend([max_handle, min_handle])
    labels.extend(["Max value in run", "Min value in run"])

    g._legend.remove()

    g.fig.legend(
        handles=handles,
        labels=labels,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=3,
        title=hue,
        frameon=True,
    )

    g.fig.suptitle(title, fontsize=16, weight="bold")

    plt.subplots_adjust(top=0.95, bottom=0.05)
    plt.show()

In [ ]:
def box_plot(metric, title, gain=True, col="Pathway Config", hue="dim"):
  
  g = sns.catplot(
    data=all_results_df,
    x="Pathway Config",
    y=metric,
    row="dim",
    hue="Model",  
    palette=[CustPalette.primary, CustPalette.primary_med, CustPalette.secondary, CustPalette.secondary_light],
    kind="box",
    width=0.5,
    gap=0.2,
    height=4,
    aspect=2.5
  )

box_plot("Util Cost", "Util cost")

## Utility Cost

In [ ]:
parallel_coord_plot("Util Cost", "Utility Cost Across Models per run", gain=False, col="dataset", hue="Pathway Config")

In [ ]:
parallel_coord_plot("Util Cost", "Utility Cost Across Models per run", gain=False)

## PPV Uplift

In [ ]:
parallel_coord_plot("PPV Uplift", "Min. Group PPV Uplift Across Models per run", gain=True, col="dataset", hue="Pathway Config")

In [ ]:
parallel_coord_plot("PPV Uplift", "Min. Group PPV Uplift Across Models per run", gain=True)

## Recall Uplift

In [ ]:
parallel_coord_plot("Recall Uplift", "Min. Group Recall Uplift Across Models per run", gain=True, col="dataset", hue="Pathway Config")

In [ ]:
parallel_coord_plot("Recall Uplift", "Min. Group Recall Uplift Across Models per run", gain=True)

## F1 Score Uplift

In [ ]:
parallel_coord_plot("F1 Uplift", "Min. Group F1 Uplift Across Models per run (% of baseline F1 score)", gain=True, col="dataset", hue="Pathway Config")

In [ ]:
parallel_coord_plot("F1 Uplift", "Min. Group F1 Uplift Across Models per run (% of baseline F1 score)", gain=True)

## Equality of Odds Ratio difference

In [ ]:
parallel_coord_plot("EOR Delta", "Equality of Odds Ratio Delta Across Models per run", gain=True, col="dataset", hue="Pathway Config")

In [ ]:
parallel_coord_plot("EOR Delta", "Equality of Odds Ratio Delta Across Models per run", gain=True)

# Statistical analysis

In [ ]:
tradeoff_metrics = ['Util Cost', 'PPV Uplift', 'Recall Uplift', 'F1 Uplift', 'EOR Delta']
grouped_results = all_results_df.groupby(["dataset", "Pathway Config", 'Model'])
tradeoff_results = grouped_results[['scm', 'dim', 'prev', 'type'] + tradeoff_metrics].first()

## Viable runs

In [ ]:
viable_configs = []

for (dataset, pw), group in tradeoff_results.groupby(level=["dataset","Pathway Config"]):
  models = group.index.get_level_values("Model")
  baseline_mask = (models == ModelLabels.baseline_unaware) | (models == ModelLabels.ablation)
  fair_mask = (models == ModelLabels.fair) | (models == ModelLabels.fair_unaware)

  summary = {}
  viable = False
  for m in ['F1 Uplift', 'Recall Uplift', 'PPV Uplift', 'EOR Delta']:
    best_fair_m = group[fair_mask][m].idxmax()
    best_baseline_m = group[baseline_mask][m].idxmax()
    win_by_m = group.loc[best_fair_m][m] > group.loc[best_baseline_m][m]

    summary = summary | {
      f"Best {m}": win_by_m,
    }

    viable = viable | win_by_m

  if viable:
    viable_configs.append({
      "Dataset": dataset,
      "Pathway Config": pw,
    } | summary)

viable_configs_df = pd.DataFrame(viable_configs)

def config_perc(x):
  return f"{x.astype(int).sum() / len(tradeoff_results)*100:.1f}%"

viable_configs_summary = viable_configs_df[['Best Recall Uplift', 'Best PPV Uplift', 'Best F1 Uplift', 'Best EOR Delta']].aggregate(
  ["sum", config_perc], axis=0
).T.rename(columns={"sum":'Count', "config_perc":'Perc.'})

print("How many configs achieved fairness gains in their run over all baselines?\n")
print(viable_configs_summary.to_markdown())

## Highest / Lowest Gain & Cost

In [ ]:
def metric_summary(metric, gain=True):
  work_df = tradeoff_results.copy()

  pathway_groupby = work_df.groupby(level=["dataset","Pathway Config"])

  work_df['highest'] = (
    work_df[metric] == pathway_groupby[metric].transform('max')).astype(int)

  # print(work_df['highest'].head(20))
  work_df['lowest']  = (
    work_df[metric] == pathway_groupby[metric].transform('min')).astype(int)

  work_df['negative'] = (work_df[metric] < 0).astype(int)
  
  lowest_results = []
  highest_results = []
  for pw, group in work_df.groupby(level="Pathway Config"):
    total_runs = len(group.index.get_level_values("dataset").unique())

    models = group.index.get_level_values("Model")
    s_aware_mask = models == ModelLabels.fair
    s_blind_mask = models == ModelLabels.fair_unaware
    baseline_mask = (models == ModelLabels.baseline_unaware) | (models == ModelLabels.ablation)
    either_fair_mask = s_aware_mask | s_blind_mask
    either_fair_positive_mask = either_fair_mask & ~group['negative']

    for col in ['lowest', 'highest']:
      # print(group[col])
      s_aware_pos = group[col][s_aware_mask].sum()
      s_blind_pos = group[col][s_blind_mask].sum()
      either_pos = group[col][either_fair_mask].sum()
      either_positive_pos = group[col][either_fair_mask].sum()

      perc_s_aware = (
        round(s_aware_pos / total_runs * 100, 2) if total_runs > 0 else 0.0
      )
      perc_s_blind = (
        (s_blind_pos / total_runs * 100) if total_runs > 0 else 0.0
      )
      perc_either = (
        (either_pos / total_runs * 100) if total_runs > 0 else 0.0
      )
      perc_positive_either = (
        (either_positive_pos / total_runs * 100) if total_runs > 0 else 0.0
      )

      record = {
        "Pathway Config": pw,
        "Total Runs": total_runs,
        f"{ModelLabels.fair} %": perc_s_aware,
        f"{ModelLabels.fair_unaware} %": perc_s_blind
      }
      if col == 'lowest':
        if gain:
          both_lower = group['highest'][baseline_mask].sum()
          record["All Fair Under Baseline %"] = (
            (both_lower / total_runs * 100) if total_runs > 0 else 0.0
          )
        else:
          record["Either Fair Model  %"] = perc_either
        lowest_results.append(record)
      else:
        if gain:
          record["Either Fair Model  %"] = perc_positive_either

        highest_results.append(record)
  
  lowest_results_df = pd.DataFrame(lowest_results)
  highest_results_df = pd.DataFrame(highest_results)

  return lowest_results_df, highest_results_df

In [ ]:
heatmap_cols = [f"{ModelLabels.fair} %", f"{ModelLabels.fair_unaware} %"]
lowest_util_cost, highest_util_cost = metric_summary("Util Cost", gain=False)

print("="*30)
print("LOWEST UTILITY COST")
display(heatmap_df(lowest_util_cost, heatmap_cols + ["Either Fair Model  %"], cmap="YlGn"))
print("-"*10)
print("HIGHEST UTILITY COST")
display(heatmap_df(highest_util_cost, heatmap_cols, cmap="Reds"))

In [ ]:
lowest_recall_uplift, highest_recall_uplift = metric_summary("Recall Uplift")

print("="*30)
print("LOWEST RECALL UPLIFT")
display(heatmap_df(lowest_recall_uplift, heatmap_cols + ["All Fair Under Baseline %"], cmap="Reds"))
print("-"*10)
print("HIGHEST RECALL UPLIFT")
display(heatmap_df(highest_recall_uplift, heatmap_cols + ["Either Fair Model  %"], cmap="YlGn"))

In [ ]:
lowest_ppv_uplift, highest_ppv_uplift = metric_summary("PPV Uplift")

print("="*30)
print("LOWEST PPV UPLIFT")
display(heatmap_df(lowest_ppv_uplift, heatmap_cols + ["All Fair Under Baseline %"], cmap="Reds"))
print("-"*10)
print("HIGHEST PPV UPLIFT")
display(heatmap_df(highest_ppv_uplift, heatmap_cols + ["Either Fair Model  %"], cmap="YlGn"))

In [ ]:
lowest_f1_uplift, highest_f1_uplift = metric_summary("F1 Uplift")

print("="*30)
print("LOWEST F1 UPLIFT")
display(heatmap_df(lowest_f1_uplift, heatmap_cols + ["All Fair Under Baseline %"], cmap="Reds"))
print("-"*10)
print("HIGHEST F1 UPLIFT")
display(heatmap_df(highest_f1_uplift, heatmap_cols + ["Either Fair Model  %"], cmap="YlGn"))

In [ ]:
lowest_eor_delta, highest_eor_delta = metric_summary("EOR Delta")

print("="*30)
print("LOWEST EQUALITY OF ODDS RATIO DIFFERENCE")
display(heatmap_df(lowest_eor_delta, heatmap_cols + ["All Fair Under Baseline %"], cmap="Reds"))
print("-"*10)
print("HIGHEST EQUALITY OF ODDS RATIO DIFFERENCE")
display(heatmap_df(highest_eor_delta, heatmap_cols + ["Either Fair Model  %"], cmap="YlGn"))